# Exercise 19: Queries Quality and Percentage

**Source:** [LeetCode 1211](https://leetcode.com/problems/queries-quality-and-percentage/description/?envType=study-plan-v2&envId=top-sql-50) (Easy).

Table: `Queries`
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| query_name  | varchar |
| result      | varchar |
| position    | int     |
| rating      | int     |
+-------------+---------+
```
- The table has no primary key and may have duplicate rows.
- A pair `(query_name, result)` is query and its result.
- `position` is the position in the database (between `1` and `500`).
- The `rating` of the query is between `1` and `5`.

A query is called a *poor query* if the rating is below 3.
The *poor query percentage* is the percentage of all poor queries between all queries.
The *quality* is the query rating divided by the position.

**Task.**
Find the **average quality** and the `poor_query_percentage` rounded to two decimals.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_19 WITH (FORCE);
%sql CREATE DATABASE exercise_19;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_19 --alias exercise_19

Connecting and switching to connection 'exercise_19'

#### Fill the database with content

In [4]:
%%sql
DROP TABLE IF EXISTS Queries;

CREATE TABLE Queries (
    query_name VARCHAR NOT NULL,
    result VARCHAR NOT NULL,
    position INTEGER NOT NULL,
    rating INTEGER NOT NULL
);

INSERT INTO Queries (query_name, result, position, rating)
VALUES
('Dog', 'Golden Retriever', 1, 5),
('Dog', 'German Shepherd', 2, 5),
('Dog', 'Mule', 200, 1),
('Cat', 'Shirazi', 5, 2),
('Cat', 'Siamese', 3, 3),
('Cat', 'Sphynx', 7, 4);

Running query in 'exercise_19'

6 rows affected.

++
||
++
++

#### Solve the exercise

A query is called a *poor query* if the rating is below 3.
The *poor query percentage* is the percentage of all poor queries between all queries.
The *quality* is the query rating divided by the position.

**Task.**
Find the **average quality** and the `poor_query_percentage` rounded to two decimals.

In [5]:
%%sql
SELECT
    query_name,
    ROUND(SUM(rating::NUMERIC/position)::NUMERIC/COUNT(*),2) AS quality,
    COALESCE(ROUND(SUM(CASE WHEN rating < 3 THEN 1 END)::NUMERIC/COUNT(*) * 100,2),0) AS poor_query_percentage
FROM
    Queries
GROUP BY
    query_name

Running query in 'exercise_19'

2 rows affected.

query_name,quality,poor_query_percentage
Cat,0.66,33.33
Dog,2.50,33.33


#### Explanation
- One needs to group by the `query_name`.
- The `quality` is the sum over `rating/position` (making sure one does a float division).
- The `poor_query_percentage` requires counting over all ratings with `rating` lower than 3. This is done with `CASE WHEN ... THEN ... END`. The result may be `NULL` if there is not poor query. Hence `COALESCE(_ , 0)` ensures to return `0` in that case.

#### Close connection

In [6]:
%sql --close exercise_19
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_19;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)